# Computation Graph Basics — Hands-On Practice

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/03_ONNX_Architecture_and_Internals/01_Computation_Graph_Basics/Computation_Graph_Basics_Apply.ipynb)

**Build, analyze, visualize, and benchmark real ONNX computation graphs**

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | [Setup and Imports](#1-setup-and-imports) | Environment preparation |
| 2 | [Building ONNX Graphs from Scratch](#2-building-onnx-graphs-from-scratch) | `onnx.helper` API |
| 3 | [Topological Sort on ONNX Graphs](#3-implementing-topological-sort-on-onnx-graphs) | Custom sort implementation |
| 4 | [Extracting Graph Properties](#4-extracting-graph-properties) | Node count, depth, width |
| 5 | [Critical Path Computation](#5-computing-the-critical-path) | Longest path analysis |
| 6 | [Visualizing ONNX Graphs](#6-visualizing-onnx-graphs-with-networkx) | NetworkX diagrams |
| 7 | [Multi-Branch Architectures](#7-multi-branch-architectures) | ResNet skip connections |
| 8 | [Comparing Graph Structures](#8-comparing-graph-structures) | Side-by-side analysis |
| 9 | [ONNX Runtime Execution](#9-running-through-onnx-runtime) | Inference + verification |
| 10 | [Performance: Shallow vs Deep](#10-performance-measurement-shallow-vs-deep-graphs) | Benchmarking |

## 1. Setup and Imports

We begin by importing all the libraries needed throughout this notebook. The core tools are:

- **`onnx`** and **`onnx.helper`**: For constructing ONNX `ModelProto` and `GraphProto` objects programmatically. The helper module provides factory functions like `make_node`, `make_graph`, and `make_model` that abstract away the raw protobuf construction.

- **`onnxruntime`**: The high-performance inference engine from Microsoft. We use it to execute our hand-built graphs and verify that the computation produces correct results.

- **`networkx`**: A Python graph library that we use to build a parallel in-memory representation of the ONNX graph for analysis and visualization. While ONNX stores edges implicitly (via tensor name matching), NetworkX gives us explicit graph algorithms like topological sort, longest path, and connectivity analysis.

- **`matplotlib`**: For rendering computation graph visualizations, bar charts comparing architectures, and performance benchmarking plots.

In [ ]:
import time
from collections import deque, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

import onnx
from onnx import TensorProto, helper, numpy_helper
from onnx.checker import check_model
import onnxruntime as ort

print(f'ONNX version:        {onnx.__version__}')
print(f'ONNX Runtime version: {ort.__version__}')
print(f'NetworkX version:     {nx.__version__}')

## 2. Building ONNX Graphs from Scratch

The `onnx.helper` module provides a clean API for constructing computation graphs without dealing with raw protobuf messages. The workflow follows a bottom-up pattern:

1. **Create initializers** (constant tensors for weights and biases) using `numpy_helper.from_array()`.
2. **Define input/output specifications** using `helper.make_tensor_value_info()` with shape and type.
3. **Build operator nodes** using `helper.make_node()` with op type, input names, and output names.
4. **Assemble the graph** using `helper.make_graph()` which bundles nodes, inputs, outputs, and initializers.
5. **Wrap in a model** using `helper.make_model()` with an opset declaration.
6. **Validate** using `onnx.checker.check_model()` to verify structural correctness.

The key insight is that **edges are implicit**: when node A produces a tensor named `h1` and node B lists `h1` as an input, the edge $A \to B$ is established. This SSA (Static Single Assignment) discipline means each tensor name must be produced by exactly one source—either a node output, a graph input, or an initializer.

### Example: Three-Layer MLP

We build a three-layer MLP computing $y = W_3 \cdot \text{ReLU}(W_2 \cdot \text{ReLU}(W_1 x + b_1) + b_2) + b_3$. This graph has 9 operator nodes (3 MatMul + 3 Add + 2 ReLU + 1 final Add for the output bias) and 6 initializer tensors.

In [ ]:
def build_mlp(layer_dims, graph_name='mlp'):
    """Build an ONNX MLP with given layer dimensions.

    Args:
        layer_dims: list of ints, e.g. [4, 8, 6, 3] for 4->8->6->3
        graph_name: name for the ONNX graph

    Returns:
        onnx.ModelProto
    """
    rng = np.random.default_rng(42)
    nodes = []
    initializers = []
    prev_output = 'X'

    for i in range(len(layer_dims) - 1):
        d_in, d_out = layer_dims[i], layer_dims[i + 1]
        w_name = f'W{i}'
        b_name = f'b{i}'
        mm_out = f'mm{i}'
        add_out = f'add{i}'
        is_last = (i == len(layer_dims) - 2)

        W = rng.standard_normal((d_in, d_out)).astype(np.float32)
        b = rng.standard_normal((d_out,)).astype(np.float32)
        initializers.append(numpy_helper.from_array(W, name=w_name))
        initializers.append(numpy_helper.from_array(b, name=b_name))

        nodes.append(helper.make_node('MatMul', [prev_output, w_name],
                                      [mm_out], name=f'fc{i}_matmul'))
        final_name = 'Y' if is_last else add_out
        nodes.append(helper.make_node('Add', [mm_out, b_name],
                                      [final_name], name=f'fc{i}_add'))

        if not is_last:
            relu_out = f'relu{i}'
            nodes.append(helper.make_node('Relu', [add_out],
                                          [relu_out], name=f'relu{i}'))
            prev_output = relu_out

    X = helper.make_tensor_value_info('X', TensorProto.FLOAT,
                                      [None, layer_dims[0]])
    Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT,
                                      [None, layer_dims[-1]])

    graph = helper.make_graph(nodes, graph_name, [X], [Y], initializers)
    model = helper.make_model(graph,
                              opset_imports=[helper.make_opsetid('', 18)])
    check_model(model)
    return model

# Build a 3-layer MLP: 4 -> 8 -> 6 -> 3
mlp_model = build_mlp([4, 8, 6, 3], 'three_layer_mlp')

print(f'Graph name: {mlp_model.graph.name}')
print(f'Nodes: {len(mlp_model.graph.node)}')
print(f'Initializers: {len(mlp_model.graph.initializer)}')
print(f'Inputs: {[i.name for i in mlp_model.graph.input]}')
print(f'Outputs: {[o.name for o in mlp_model.graph.output]}')
print()
print('Node list:')
for i, node in enumerate(mlp_model.graph.node):
    print(f'  [{i}] {node.name:15s} {node.op_type:8s} '
          f'{list(node.input)} -> {list(node.output)}')

## 3. Implementing Topological Sort on ONNX Graphs

While ONNX stores nodes in topological order by convention, we implement Kahn's algorithm on the ONNX graph structure to:

1. **Verify** that the stored order is indeed a valid topological ordering.
2. **Understand** how the implicit edge structure (SSA naming) translates to explicit graph dependencies.
3. **Detect** potential cycles or ordering violations that might indicate a corrupt model.

The key challenge in applying Kahn's algorithm to ONNX is reconstructing the **explicit edge set** from the implicit SSA representation. We build a mapping from tensor names to their producers (either nodes or graph inputs/initializers), then use this to determine the predecessor relationship between nodes.

Our implementation computes the in-degree of each node by counting how many of its input tensors come from other nodes (as opposed to graph inputs or initializers, which are "free" sources). The algorithm then iteratively removes zero-in-degree nodes, decrements the in-degrees of their successors, and accumulates the topological ordering.

In [ ]:
def onnx_topological_sort(graph_proto):
    """Kahn's topological sort on an ONNX GraphProto.

    Returns:
        ordering: list of NodeProto in topological order
        edge_list: list of (producer_node_name, consumer_node_name) tuples
    """
    free_names = set()
    for inp in graph_proto.input:
        free_names.add(inp.name)
    for init in graph_proto.initializer:
        free_names.add(init.name)

    tensor_to_node = {}
    for node in graph_proto.node:
        for out in node.output:
            tensor_to_node[out] = node.name

    node_map = {node.name: node for node in graph_proto.node}
    successors = defaultdict(set)
    in_degree = {node.name: 0 for node in graph_proto.node}
    edge_list = []

    for node in graph_proto.node:
        for inp_name in node.input:
            if inp_name and inp_name not in free_names:
                producer = tensor_to_node.get(inp_name)
                if producer and producer != node.name:
                    if node.name not in successors[producer]:
                        successors[producer].add(node.name)
                        in_degree[node.name] += 1
                        edge_list.append((producer, node.name))

    queue = deque(n for n in in_degree if in_degree[n] == 0)
    ordering = []

    while queue:
        name = queue.popleft()
        ordering.append(node_map[name])
        for succ in successors[name]:
            in_degree[succ] -= 1
            if in_degree[succ] == 0:
                queue.append(succ)

    if len(ordering) != len(graph_proto.node):
        raise ValueError(
            f'Cycle detected! Processed {len(ordering)}/{len(graph_proto.node)} nodes.'
        )

    return ordering, edge_list


# Apply to our MLP model
sorted_nodes, edges = onnx_topological_sort(mlp_model.graph)

print('Topological ordering (Kahn\'s algorithm):')
for i, node in enumerate(sorted_nodes):
    print(f'  sigma({node.name}) = {i+1}: '
          f'{node.op_type}({list(node.input)}) -> {list(node.output)}')

print(f'\nEdges ({len(edges)} total):')
for u, v in edges:
    print(f'  {u} -> {v}')

# Verify: stored order matches topological order
stored_order = [n.name for n in mlp_model.graph.node]
computed_order = [n.name for n in sorted_nodes]
print(f'\nStored order valid? '
      f'{stored_order == computed_order or "(different but still valid)"}')

## 4. Extracting Graph Properties

Now we build a comprehensive analysis tool that extracts key structural properties from any ONNX graph. These properties directly influence optimization strategies and runtime performance:

- **Node count** ($|V|$): Total operator invocations = total work.
- **Edge count** ($|E|$): Data dependencies = communication overhead.
- **Depth** (longest path): Minimum sequential steps = latency lower bound.
- **Width** (max level size): Maximum parallelism = throughput upper bound.
- **Operator distribution**: Which ops dominate the graph? This guides fusion priorities.
- **In-degree / out-degree statistics**: High fan-in nodes are synchronization bottlenecks; high fan-out nodes are broadcast points.

The function `analyze_onnx_graph()` below converts an ONNX `GraphProto` into a NetworkX `DiGraph`, then leverages NetworkX's graph algorithms to compute these metrics efficiently.

In [ ]:
def onnx_to_networkx(graph_proto):
    """Convert an ONNX GraphProto to a NetworkX DiGraph."""
    free_names = set()
    for inp in graph_proto.input:
        free_names.add(inp.name)
    for init in graph_proto.initializer:
        free_names.add(init.name)

    tensor_to_node = {}
    G = nx.DiGraph()

    for node in graph_proto.node:
        G.add_node(node.name, op_type=node.op_type)
        for out in node.output:
            tensor_to_node[out] = node.name

    for node in graph_proto.node:
        for inp_name in node.input:
            if inp_name and inp_name not in free_names:
                producer = tensor_to_node.get(inp_name)
                if producer and producer != node.name:
                    G.add_edge(producer, node.name, tensor=inp_name)

    return G


def analyze_onnx_graph(model, name='Model'):
    """Comprehensive structural analysis of an ONNX model."""
    graph = model.graph
    G = onnx_to_networkx(graph)

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()

    # Level decomposition
    levels = {}
    for node in nx.topological_sort(G):
        preds = list(G.predecessors(node))
        levels[node] = 0 if not preds else max(levels[p] for p in preds) + 1
    level_groups = defaultdict(list)
    for node, lv in levels.items():
        level_groups[lv].append(node)

    depth = max(levels.values()) if levels else 0
    width = max(len(v) for v in level_groups.values()) if level_groups else 0

    # Degree stats
    in_degs = [d for _, d in G.in_degree()]
    out_degs = [d for _, d in G.out_degree()]

    # Operator distribution
    op_counts = defaultdict(int)
    for node in graph.node:
        op_counts[node.op_type] += 1

    # Longest path
    longest = nx.dag_longest_path(G) if n_nodes > 0 else []

    results = {
        'name': name,
        'nodes': n_nodes,
        'edges': n_edges,
        'depth': depth,
        'width': width,
        'levels': dict(level_groups),
        'longest_path': longest,
        'op_counts': dict(op_counts),
        'avg_in_degree': np.mean(in_degs) if in_degs else 0,
        'avg_out_degree': np.mean(out_degs) if out_degs else 0,
        'max_in_degree': max(in_degs) if in_degs else 0,
        'max_out_degree': max(out_degs) if out_degs else 0,
        'n_initializers': len(graph.initializer),
        'networkx_graph': G
    }

    print(f'\n{"=" * 50}')
    print(f' Graph Analysis: {name}')
    print(f'{"=" * 50}')
    print(f'  |V| (nodes):        {n_nodes}')
    print(f'  |E| (edges):        {n_edges}')
    print(f'  Initializers:       {results["n_initializers"]}')
    print(f'  Depth:              {depth}')
    print(f'  Width:              {width}')
    print(f'  Avg parallelism:    {n_nodes / (depth + 1):.2f}')
    print(f'  Max speedup:        {n_nodes / (depth + 1):.2f}x')
    print(f'  Avg in-degree:      {results["avg_in_degree"]:.2f}')
    print(f'  Max in-degree:      {results["max_in_degree"]}')
    print(f'  Avg out-degree:     {results["avg_out_degree"]:.2f}')
    print(f'  Max out-degree:     {results["max_out_degree"]}')
    print(f'  Longest path:       {" -> ".join(longest)}')
    print(f'  Operator breakdown:')
    for op, count in sorted(op_counts.items(), key=lambda x: -x[1]):
        pct = 100 * count / n_nodes
        print(f'    {op:15s}: {count:3d} ({pct:5.1f}%)')

    return results


mlp_analysis = analyze_onnx_graph(mlp_model, 'Three-Layer MLP')

## 5. Computing the Critical Path

The **critical path** determines the minimum possible latency of executing the graph, even with unlimited parallelism. We implement two versions of the critical path computation:

1. **Unweighted**: Each operator counts as 1 step. The critical path length equals the graph depth $+ 1$.
2. **Weighted**: Each operator has an estimated cost (e.g., MatMul costs more than Add). This gives a more realistic estimate of actual execution time.

The algorithm uses dynamic programming in topological order. For each node $v$, we compute:

$$\text{dist}(v) = w(v) + \max_{(u,v) \in E} \text{dist}(u)$$

where $w(v)$ is the weight (cost) of node $v$. The critical path is then recovered by backtracking from the node with maximum distance.

In [ ]:
def compute_critical_path_onnx(model, op_weights=None):
    """Compute the critical path through an ONNX model's graph.

    Args:
        model: ONNX ModelProto
        op_weights: dict mapping op_type -> cost (default: all 1)

    Returns:
        path: list of node names on the critical path
        total_cost: critical path length
        all_distances: dict of node_name -> distance from source
    """
    G = onnx_to_networkx(model.graph)

    if op_weights is None:
        weights = {n: 1 for n in G.nodes()}
    else:
        weights = {}
        for node in model.graph.node:
            weights[node.name] = op_weights.get(node.op_type, 1)

    dist = {}
    pred = {}

    for node in nx.topological_sort(G):
        predecessors = list(G.predecessors(node))
        if not predecessors:
            dist[node] = weights[node]
            pred[node] = None
        else:
            best = max(predecessors, key=lambda p: dist[p])
            dist[node] = weights[node] + dist[best]
            pred[node] = best

    end = max(dist, key=dist.get)
    path = []
    cur = end
    while cur is not None:
        path.append(cur)
        cur = pred[cur]
    path.reverse()

    return path, dist[end], dist


# Unweighted critical path
cp, cpl, dists = compute_critical_path_onnx(mlp_model)
print(f'Unweighted critical path: {" -> ".join(cp)}')
print(f'Critical path length: {cpl}')
total_work = len(mlp_model.graph.node)
print(f'Total work: {total_work}')
print(f'Average parallelism: {total_work / cpl:.2f}')

# Weighted critical path (MatMul is expensive)
weights = {'MatMul': 10, 'Add': 1, 'Relu': 1}
cp_w, cpl_w, dists_w = compute_critical_path_onnx(mlp_model, weights)
print(f'\nWeighted critical path: {" -> ".join(cp_w)}')
print(f'Weighted CPL: {cpl_w}')
total_weighted = sum(weights.get(n.op_type, 1) for n in mlp_model.graph.node)
print(f'Total weighted work: {total_weighted}')
print(f'Weighted avg parallelism: {total_weighted / cpl_w:.2f}')

## 6. Visualizing ONNX Graphs with NetworkX

Visualization is essential for understanding and debugging computation graphs. We build a reusable visualization function that:

1. Converts the ONNX graph to a NetworkX `DiGraph`.
2. Computes a **layered layout** based on topological levels (so data flows top-to-bottom).
3. **Color-codes nodes** by operator type for quick visual identification.
4. Annotates edges with tensor names for data flow tracing.
5. Optionally highlights the **critical path** to show the latency-determining chain.

The layout algorithm assigns each node to its topological level (longest path from any source), then distributes nodes within each level horizontally. This produces clean, readable visualizations for graphs of moderate size (up to ~50 nodes).

In [ ]:
OP_COLORS = {
    'MatMul': '#1565C0', 'Gemm': '#1565C0',
    'Add': '#2E7D32', 'Sub': '#2E7D32',
    'Relu': '#E65100', 'Sigmoid': '#E65100', 'Tanh': '#E65100',
    'Softmax': '#AD1457',
    'Conv': '#6A1B9A',
    'BatchNormalization': '#00838F',
    'Concat': '#F9A825',
    'Reshape': '#546E7A', 'Flatten': '#546E7A',
}
DEFAULT_COLOR = '#78909C'


def visualize_onnx_graph(model, title=None, highlight_critical=False,
                         figsize=(14, 8), op_weights=None):
    """Visualize an ONNX model's computation graph."""
    G = onnx_to_networkx(model.graph)
    if G.number_of_nodes() == 0:
        print('Empty graph, nothing to visualize.')
        return

    # Compute layered positions
    levels = {}
    for node in nx.topological_sort(G):
        preds = list(G.predecessors(node))
        levels[node] = 0 if not preds else max(levels[p] for p in preds) + 1
    level_groups = defaultdict(list)
    for node, lv in levels.items():
        level_groups[lv].append(node)

    pos = {}
    for lv, nodes in level_groups.items():
        for i, node in enumerate(nodes):
            pos[node] = (i - (len(nodes) - 1) / 2.0, -lv)

    # Node colors by op type
    op_type_map = {n.name: n.op_type for n in model.graph.node}
    node_colors = [OP_COLORS.get(op_type_map.get(n, ''), DEFAULT_COLOR)
                   for n in G.nodes()]

    # Labels: show op_type instead of node name
    labels = {n: op_type_map.get(n, n) for n in G.nodes()}

    fig, ax = plt.subplots(1, 1, figsize=figsize)

    edge_colors = ['#999999'] * G.number_of_edges()
    edge_widths = [1.5] * G.number_of_edges()

    if highlight_critical:
        cp, _, _ = compute_critical_path_onnx(model, op_weights)
        cp_set = set(cp)
        cp_edges = set(zip(cp[:-1], cp[1:]))
        node_colors = ['#FF1744' if n in cp_set else c
                       for n, c in zip(G.nodes(), node_colors)]
        edge_list = list(G.edges())
        edge_colors = ['#FF1744' if e in cp_edges else '#CCCCCC'
                       for e in edge_list]
        edge_widths = [3.0 if e in cp_edges else 1.0 for e in edge_list]

    nx.draw(G, pos, ax=ax, labels=labels, node_color=node_colors,
            node_size=2000, font_size=9, font_weight='bold',
            font_color='white', edge_color=edge_colors, width=edge_widths,
            arrows=True, arrowsize=18, node_shape='s')

    # Legend
    seen_ops = set(op_type_map.values())
    legend_items = []
    for op in sorted(seen_ops):
        color = OP_COLORS.get(op, DEFAULT_COLOR)
        legend_items.append(mpatches.Patch(color=color, label=op))
    if highlight_critical:
        legend_items.append(mpatches.Patch(color='#FF1744',
                                           label='Critical Path'))
    ax.legend(handles=legend_items, loc='upper left', fontsize=9)

    if title:
        ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.show()


visualize_onnx_graph(mlp_model,
                     title='Three-Layer MLP Computation Graph')
visualize_onnx_graph(mlp_model,
                     title='Three-Layer MLP - Critical Path Highlighted',
                     highlight_critical=True)

## 7. Multi-Branch Architectures

Real-world neural networks often use **multi-branch** topologies that go beyond simple sequential chains. Two of the most influential patterns are:

1. **Residual connections** (ResNet): The input to a block is added to the output of that block, creating a "skip" or "shortcut" connection. Formally, if the block computes $\mathcal{F}(x)$, the output is $y = \mathcal{F}(x) + x$. This creates a diamond pattern in the DAG.

2. **Multi-branch concatenation** (Inception): The input is processed by several parallel branches (e.g., $1 \times 1$ conv, $3 \times 3$ conv, max pooling), and the results are concatenated along the channel dimension.

Let us build both patterns as ONNX graphs and analyze their structural differences.

In [ ]:
def build_resnet_block(in_channels=16, mid_channels=16, name_prefix='res'):
    """Build a ResNet-like residual block as an ONNX model.

    Architecture: x -> Conv -> Relu -> Conv -> Add(x) -> Relu -> y
    """
    rng = np.random.default_rng(42)

    W1 = rng.standard_normal((mid_channels, in_channels, 3, 3)).astype(np.float32)
    W2 = rng.standard_normal((in_channels, mid_channels, 3, 3)).astype(np.float32)

    nodes = [
        helper.make_node('Conv', ['X', 'W1'], ['conv1_out'],
                         name=f'{name_prefix}_conv1',
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node('Relu', ['conv1_out'], ['relu1_out'],
                         name=f'{name_prefix}_relu1'),
        helper.make_node('Conv', ['relu1_out', 'W2'], ['conv2_out'],
                         name=f'{name_prefix}_conv2',
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node('Add', ['conv2_out', 'X'], ['residual_out'],
                         name=f'{name_prefix}_skip_add'),
        helper.make_node('Relu', ['residual_out'], ['Y'],
                         name=f'{name_prefix}_relu2'),
    ]

    X = helper.make_tensor_value_info('X', TensorProto.FLOAT,
                                      [1, in_channels, 8, 8])
    Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT,
                                      [1, in_channels, 8, 8])

    initializers = [
        numpy_helper.from_array(W1, name='W1'),
        numpy_helper.from_array(W2, name='W2'),
    ]

    graph = helper.make_graph(nodes, 'resnet_block', [X], [Y], initializers)
    model = helper.make_model(graph,
                              opset_imports=[helper.make_opsetid('', 18)])
    check_model(model)
    return model


def build_inception_block(in_channels=16, name_prefix='inc'):
    """Build an Inception-like multi-branch block as an ONNX model.

    Architecture: x -> [Conv1x1 | Conv3x3 | MaxPool+Conv1x1] -> Concat -> y
    """
    rng = np.random.default_rng(42)
    br1_ch, br2_ch, br3_ch = 8, 8, 8

    W_1x1 = rng.standard_normal((br1_ch, in_channels, 1, 1)).astype(np.float32)
    W_3x3 = rng.standard_normal((br2_ch, in_channels, 3, 3)).astype(np.float32)
    W_pool = rng.standard_normal((br3_ch, in_channels, 1, 1)).astype(np.float32)

    nodes = [
        helper.make_node('Conv', ['X', 'W_1x1'], ['br1_out'],
                         name=f'{name_prefix}_conv1x1',
                         kernel_shape=[1, 1]),
        helper.make_node('Conv', ['X', 'W_3x3'], ['br2_out'],
                         name=f'{name_prefix}_conv3x3',
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node('MaxPool', ['X'], ['pool_out'],
                         name=f'{name_prefix}_maxpool',
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node('Conv', ['pool_out', 'W_pool'], ['br3_out'],
                         name=f'{name_prefix}_pool_conv',
                         kernel_shape=[1, 1]),
        helper.make_node('Concat', ['br1_out', 'br2_out', 'br3_out'], ['Y'],
                         name=f'{name_prefix}_concat', axis=1),
    ]

    out_channels = br1_ch + br2_ch + br3_ch
    X = helper.make_tensor_value_info('X', TensorProto.FLOAT,
                                      [1, in_channels, 8, 8])
    Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT,
                                      [1, out_channels, 8, 8])

    initializers = [
        numpy_helper.from_array(W_1x1, name='W_1x1'),
        numpy_helper.from_array(W_3x3, name='W_3x3'),
        numpy_helper.from_array(W_pool, name='W_pool'),
    ]

    graph = helper.make_graph(nodes, 'inception_block', [X], [Y], initializers)
    model = helper.make_model(graph,
                              opset_imports=[helper.make_opsetid('', 18)])
    check_model(model)
    return model


resnet_model = build_resnet_block()
inception_model = build_inception_block()

print('=== ResNet Block ===')
for i, n in enumerate(resnet_model.graph.node):
    print(f'  [{i}] {n.name:20s} {n.op_type:8s} {list(n.input)} -> {list(n.output)}')

print('\n=== Inception Block ===')
for i, n in enumerate(inception_model.graph.node):
    print(f'  [{i}] {n.name:20s} {n.op_type:8s} {list(n.input)} -> {list(n.output)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, model, title in [
    (axes[0], resnet_model, 'ResNet Block (Skip Connection)'),
    (axes[1], inception_model, 'Inception Block (Multi-Branch)')
]:
    G = onnx_to_networkx(model.graph)
    levels = {}
    for node in nx.topological_sort(G):
        preds = list(G.predecessors(node))
        levels[node] = 0 if not preds else max(levels[p] for p in preds) + 1
    lg = defaultdict(list)
    for node, lv in levels.items():
        lg[lv].append(node)

    pos = {}
    for lv, nodes in lg.items():
        for i, node in enumerate(nodes):
            pos[node] = (i - (len(nodes) - 1) / 2.0, -lv)

    op_map = {n.name: n.op_type for n in model.graph.node}
    colors = [OP_COLORS.get(op_map.get(n, ''), DEFAULT_COLOR) for n in G.nodes()]
    labels = {n: op_map.get(n, n) for n in G.nodes()}

    nx.draw(G, pos, ax=ax, labels=labels, node_color=colors,
            node_size=2200, font_size=9, font_weight='bold',
            font_color='white', edge_color='#666', arrows=True,
            arrowsize=15, node_shape='s')

    depth = max(levels.values()) if levels else 0
    width = max(len(v) for v in lg.values()) if lg else 0
    ax.set_title(f'{title}\n|V|={G.number_of_nodes()}, |E|={G.number_of_edges()}, '
                 f'depth={depth}, width={width}', fontsize=11)

plt.tight_layout()
plt.show()

## 8. Comparing Graph Structures

Let us build several different architectures and compare their structural properties side by side. This comparison reveals how architectural choices affect the graph-theoretic properties that influence optimization and execution strategies.

We compare four architectures:
1. **Shallow MLP** (2 layers): Simple feedforward, minimal depth.
2. **Deep MLP** (6 layers): Deep feedforward, long sequential chain.
3. **ResNet block**: Skip connection introduces branching.
4. **Inception block**: Multi-branch introduces high width.

The key metrics for comparison are:
- **Depth** vs **Width**: Deep networks have long critical paths; wide networks have more parallelism.
- **Average parallelism** ($|V| / (\text{depth} + 1)$): How many operations can run simultaneously on average.
- **Edge density** ($|E| / (|V| \cdot (|V| - 1))$): How connected the graph is.

In [ ]:
# Build all architectures
models = {
    'Shallow MLP\n(2 layers)': build_mlp([4, 8, 3], 'shallow_mlp'),
    'Medium MLP\n(4 layers)': build_mlp([4, 8, 8, 6, 3], 'medium_mlp'),
    'Deep MLP\n(6 layers)': build_mlp([4, 8, 8, 8, 8, 6, 3], 'deep_mlp'),
    'ResNet\nBlock': resnet_model,
    'Inception\nBlock': inception_model,
}

analyses = {}
for name, model in models.items():
    analyses[name] = analyze_onnx_graph(model, name.replace('\n', ' '))

# Comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
names = list(analyses.keys())
x_pos = np.arange(len(names))
bar_colors = ['#1565C0', '#2E7D32', '#E65100', '#AD1457', '#6A1B9A']

# Plot 1: Node and Edge counts
ax = axes[0, 0]
nodes_vals = [analyses[n]['nodes'] for n in names]
edges_vals = [analyses[n]['edges'] for n in names]
w = 0.35
ax.bar(x_pos - w/2, nodes_vals, w, label='|V| (nodes)', color='#1565C0')
ax.bar(x_pos + w/2, edges_vals, w, label='|E| (edges)', color='#E65100')
ax.set_xticks(x_pos)
ax.set_xticklabels(names, fontsize=8)
ax.set_ylabel('Count')
ax.set_title('Graph Size: Nodes and Edges')
ax.legend()

# Plot 2: Depth and Width
ax = axes[0, 1]
depth_vals = [analyses[n]['depth'] for n in names]
width_vals = [analyses[n]['width'] for n in names]
ax.bar(x_pos - w/2, depth_vals, w, label='Depth', color='#AD1457')
ax.bar(x_pos + w/2, width_vals, w, label='Width', color='#2E7D32')
ax.set_xticks(x_pos)
ax.set_xticklabels(names, fontsize=8)
ax.set_ylabel('Value')
ax.set_title('Depth vs Width')
ax.legend()

# Plot 3: Average parallelism
ax = axes[1, 0]
par_vals = [analyses[n]['nodes'] / (analyses[n]['depth'] + 1) for n in names]
bars = ax.bar(x_pos, par_vals, color=bar_colors)
ax.set_xticks(x_pos)
ax.set_xticklabels(names, fontsize=8)
ax.set_ylabel('Avg Parallelism (|V|/(depth+1))')
ax.set_title('Average Parallelism')
for bar, val in zip(bars, par_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)

# Plot 4: Operator distribution
ax = axes[1, 1]
all_ops = set()
for a in analyses.values():
    all_ops.update(a['op_counts'].keys())
all_ops = sorted(all_ops)
bottom = np.zeros(len(names))
op_colors = plt.cm.Set3(np.linspace(0, 1, len(all_ops)))
for op, color in zip(all_ops, op_colors):
    vals = [analyses[n]['op_counts'].get(op, 0) for n in names]
    ax.bar(x_pos, vals, bottom=bottom, label=op, color=color)
    bottom += vals
ax.set_xticks(x_pos)
ax.set_xticklabels(names, fontsize=8)
ax.set_ylabel('Count')
ax.set_title('Operator Distribution')
ax.legend(fontsize=7, ncol=2)

fig.suptitle('Architecture Comparison: Graph Properties', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 9. Running Through ONNX Runtime

Now we verify that our hand-built ONNX graphs actually compute correct results by running them through **ONNX Runtime** (ORT). This serves two purposes:

1. **Correctness verification**: We compare ORT's output against a manual NumPy computation to ensure the graph is semantically correct.
2. **Understanding execution order**: ORT internally performs its own topological sort and may reorder nodes for optimization. We can inspect whether the runtime respects the graph's dependency structure.

For each model, we generate random input data with the correct shape, run inference with ORT, and compare against the expected output computed manually with NumPy.

In [ ]:
def run_and_verify(model, input_data, expected=None, model_name='Model'):
    """Run model through ORT and optionally verify against expected output."""
    sess = ort.InferenceSession(model.SerializeToString(),
                                providers=['CPUExecutionProvider'])

    input_name = sess.get_inputs()[0].name
    output_name = sess.get_outputs()[0].name

    result = sess.run([output_name], {input_name: input_data})

    print(f'=== {model_name} ===')
    print(f'  Input:  shape={input_data.shape}, dtype={input_data.dtype}')
    print(f'  Output: shape={result[0].shape}, dtype={result[0].dtype}')
    print(f'  Output sample: {result[0].flatten()[:5]}...')

    if expected is not None:
        max_diff = np.max(np.abs(result[0] - expected))
        match = np.allclose(result[0], expected, atol=1e-5)
        print(f'  Max diff from expected: {max_diff:.2e}')
        print(f'  Match: {"PASS" if match else "FAIL"}')

    return result[0]


# Verify the three-layer MLP
rng = np.random.default_rng(42)
x_mlp = rng.standard_normal((2, 4)).astype(np.float32)

# Manual NumPy computation for verification
inits = {i.name: numpy_helper.to_array(i)
         for i in mlp_model.graph.initializer}

h = x_mlp
for i in range(3):
    h = h @ inits[f'W{i}'] + inits[f'b{i}']
    if i < 2:
        h = np.maximum(0, h)
expected_mlp = h

ort_result = run_and_verify(mlp_model, x_mlp, expected_mlp,
                            'Three-Layer MLP')

# Verify the ResNet block
x_res = rng.standard_normal((1, 16, 8, 8)).astype(np.float32)
res_result = run_and_verify(resnet_model, x_res, model_name='ResNet Block')

# Verify the Inception block
x_inc = rng.standard_normal((1, 16, 8, 8)).astype(np.float32)
inc_result = run_and_verify(inception_model, x_inc,
                            model_name='Inception Block')

## 10. Performance Measurement: Shallow vs Deep Graphs

A fundamental question in neural network architecture design is: **How does graph depth affect inference performance?** Deeper graphs have longer critical paths, meaning more sequential steps. But they may also allow the optimizer to find better fusion opportunities.

We conduct a controlled experiment:
- Fix the total number of parameters approximately constant.
- Vary the depth (number of layers) from 2 to 8.
- Measure inference latency for each configuration.
- Analyze how latency scales with graph depth and total node count.

This experiment illustrates the practical consequences of the theoretical **work-depth tradeoff**: deeper graphs have more sequential dependencies (higher depth $D$), which limits the parallelism achievable even with multiple processor cores.

### Theoretical Prediction

From Brent's theorem, on $P$ processors:

$$T_P \leq \frac{W - D}{P} + D$$

For our MLP experiments on a single CPU core ($P = 1$), this simplifies to $T_1 = W$, meaning latency scales linearly with total work. But since each MatMul has different dimensions across configurations, the actual work per node varies.

In [ ]:
def benchmark_model(model, input_shape, n_warmup=10, n_runs=100):
    """Benchmark inference latency of an ONNX model."""
    sess = ort.InferenceSession(model.SerializeToString(),
                                providers=['CPUExecutionProvider'])
    input_name = sess.get_inputs()[0].name
    x = np.random.randn(*input_shape).astype(np.float32)

    for _ in range(n_warmup):
        sess.run(None, {input_name: x})

    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        sess.run(None, {input_name: x})
        times.append((time.perf_counter() - start) * 1000)

    return {
        'mean_ms': np.mean(times),
        'std_ms': np.std(times),
        'median_ms': np.median(times),
        'p95_ms': np.percentile(times, 95),
        'min_ms': np.min(times),
    }


configs = [
    ('2-layer', [32, 64, 32]),
    ('3-layer', [32, 48, 48, 32]),
    ('4-layer', [32, 40, 40, 40, 32]),
    ('5-layer', [32, 36, 36, 36, 36, 32]),
    ('6-layer', [32, 34, 34, 34, 34, 34, 32]),
    ('8-layer', [32, 32, 32, 32, 32, 32, 32, 32, 32]),
]

benchmark_results = []
for name, dims in configs:
    model = build_mlp(dims, name)
    analysis = analyze_onnx_graph(model, name)
    perf = benchmark_model(model, (16, dims[0]))

    n_params = sum(np.prod(numpy_helper.to_array(i).shape)
                   for i in model.graph.initializer)

    benchmark_results.append({
        'name': name,
        'dims': dims,
        'n_layers': len(dims) - 1,
        'n_nodes': analysis['nodes'],
        'n_edges': analysis['edges'],
        'depth': analysis['depth'],
        'width': analysis['width'],
        'n_params': n_params,
        **perf
    })
    print(f'  {name}: {perf["mean_ms"]:.3f} ms (\u00b1{perf["std_ms"]:.3f}), '
          f'params={n_params}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

labels = [r['name'] for r in benchmark_results]
x_pos = np.arange(len(labels))

# Plot 1: Latency vs Depth
ax = axes[0, 0]
depths = [r['depth'] for r in benchmark_results]
means = [r['mean_ms'] for r in benchmark_results]
stds = [r['std_ms'] for r in benchmark_results]
ax.errorbar(depths, means, yerr=stds, fmt='o-', capsize=5,
            color='#1565C0', linewidth=2, markersize=8)
ax.set_xlabel('Graph Depth', fontsize=11)
ax.set_ylabel('Latency (ms)', fontsize=11)
ax.set_title('Latency vs Graph Depth', fontsize=12)
ax.grid(True, alpha=0.3)

# Plot 2: Latency vs Node Count
ax = axes[0, 1]
n_nodes = [r['n_nodes'] for r in benchmark_results]
ax.errorbar(n_nodes, means, yerr=stds, fmt='s-', capsize=5,
            color='#E65100', linewidth=2, markersize=8)
ax.set_xlabel('Node Count |V|', fontsize=11)
ax.set_ylabel('Latency (ms)', fontsize=11)
ax.set_title('Latency vs Total Work (Node Count)', fontsize=12)
ax.grid(True, alpha=0.3)

# Plot 3: Parameter count and depth
ax = axes[1, 0]
params = [r['n_params'] for r in benchmark_results]
w = 0.35
ax.bar(x_pos - w/2, params, w, label='Parameters', color='#2E7D32')
ax2 = ax.twinx()
ax2.bar(x_pos + w/2, depths, w, label='Depth', color='#AD1457', alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Parameters', fontsize=11, color='#2E7D32')
ax2.set_ylabel('Depth', fontsize=11, color='#AD1457')
ax.set_title('Parameters vs Depth', fontsize=12)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Plot 4: Throughput (ops/ms)
ax = axes[1, 1]
throughput = [r['n_nodes'] / r['mean_ms'] for r in benchmark_results]
bars = ax.bar(x_pos, throughput, color=bar_colors[:len(x_pos)])
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Throughput (ops/ms)', fontsize=11)
ax.set_title('Graph Throughput', fontsize=12)
for bar, val in zip(bars, throughput):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Performance: Shallow vs Deep Graphs',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 11. Advanced: Graph Isomorphism and Structural Fingerprinting

When comparing different ONNX models, it is useful to determine whether two computation graphs are **structurally identical** (isomorphic) up to node renaming, or to compute a **structural fingerprint** that summarizes the graph's topology.

Two graphs $G_1 = (V_1, E_1)$ and $G_2 = (V_2, E_2)$ are **isomorphic** if there exists a bijection $\phi: V_1 \to V_2$ such that $(u, v) \in E_1 \iff (\phi(u), \phi(v)) \in E_2$. For computation graphs, we additionally require that corresponding nodes have the same operator type: $\ell_1(v) = \ell_2(\phi(v))$.

We implement a simple **structural fingerprint** based on the sorted sequence of (in-degree, out-degree, op_type) tuples for all nodes, plus the depth and width. While not a complete isomorphism test, this quickly identifies structurally different graphs.

In [ ]:
def structural_fingerprint(model):
    """Compute a structural fingerprint of an ONNX model's graph."""
    G = onnx_to_networkx(model.graph)
    op_map = {n.name: n.op_type for n in model.graph.node}

    node_sigs = sorted(
        (G.in_degree(n), G.out_degree(n), op_map.get(n, 'unknown'))
        for n in G.nodes()
    )

    levels = {}
    for node in nx.topological_sort(G):
        preds = list(G.predecessors(node))
        levels[node] = 0 if not preds else max(levels[p] for p in preds) + 1

    depth = max(levels.values()) if levels else 0
    lg = defaultdict(list)
    for nd, lv in levels.items():
        lg[lv].append(nd)
    width = max(len(v) for v in lg.values()) if lg else 0

    level_profile = tuple(sorted(len(v) for v in lg.values()))

    return {
        'n_nodes': G.number_of_nodes(),
        'n_edges': G.number_of_edges(),
        'depth': depth,
        'width': width,
        'node_signatures': tuple(node_sigs),
        'level_profile': level_profile,
    }


def compare_fingerprints(fp1, fp2, name1='Graph A', name2='Graph B'):
    """Compare two structural fingerprints."""
    print(f'\nComparing {name1} vs {name2}:')
    identical = True
    for key in ['n_nodes', 'n_edges', 'depth', 'width', 'level_profile']:
        match = fp1[key] == fp2[key]
        if not match:
            identical = False
        sym = '==' if match else '!='
        print(f'  {key:18s}: {str(fp1[key]):20s} {sym} {str(fp2[key])}')

    same_sigs = fp1['node_signatures'] == fp2['node_signatures']
    print(f'  {"node_signatures":18s}: {"MATCH" if same_sigs else "DIFFER"}')
    print(f'  Structurally identical: {identical and same_sigs}')


# Compare our architectures
fp_mlp3 = structural_fingerprint(build_mlp([4, 8, 6, 3]))
fp_mlp3b = structural_fingerprint(build_mlp([4, 8, 6, 3]))
fp_mlp4 = structural_fingerprint(build_mlp([4, 8, 8, 6, 3]))
fp_res = structural_fingerprint(resnet_model)
fp_inc = structural_fingerprint(inception_model)

compare_fingerprints(fp_mlp3, fp_mlp3b, '3-Layer MLP (a)', '3-Layer MLP (b)')
compare_fingerprints(fp_mlp3, fp_mlp4, '3-Layer MLP', '4-Layer MLP')
compare_fingerprints(fp_res, fp_inc, 'ResNet Block', 'Inception Block')

## Summary

In this hands-on notebook, we built a complete toolkit for constructing, analyzing, and benchmarking ONNX computation graphs:

### Tools Built

| Function | Purpose |
|----------|--------|
| `build_mlp()` | Programmatically construct MLP ONNX models of arbitrary depth |
| `onnx_topological_sort()` | Kahn's algorithm on ONNX `GraphProto` ($O(|V| + |E|)$) |
| `onnx_to_networkx()` | Convert implicit ONNX edges to explicit NetworkX graph |
| `analyze_onnx_graph()` | Extract depth, width, degree stats, operator distribution |
| `compute_critical_path_onnx()` | Weighted/unweighted critical path via DP |
| `visualize_onnx_graph()` | Layered DAG visualization with operator coloring |
| `structural_fingerprint()` | Topology-based graph comparison |
| `benchmark_model()` | ORT inference latency measurement |

### Key Findings

1. **Graph structure directly affects performance**: Deeper graphs have longer critical paths and higher latency.
2. **Multi-branch patterns** (ResNet, Inception) increase width and parallelism compared to sequential chains.
3. **SSA naming** in ONNX creates implicit edges that can be reconstructed into explicit graph structure for analysis.
4. **Topological sort verification** confirms that ONNX stores nodes in a valid execution order.
5. **Critical path analysis** with realistic operator weights gives more accurate latency predictions than unweighted depth.

---

**Next:** [Nodes, Edges, and Tensors](../02_Nodes_Edges_and_Tensors/) — deeper dive into ONNX node semantics, tensor types, and edge representations.